## LangGraph: First Agent

A LangGraph agent is a **graph of nodes** that all read and write a shared **state** object.

- **State**: a schema (usually a `TypedDict`) describing every field the graph carries around.
- **Node**: a plain Python function `(state) -> partial_state`. It receives the *whole* state and returns the keys it wants to update.
- **Graph**: wires nodes together with edges, then `compile()`s into a runnable `app`.

Below we build the smallest possible version: one node, one edge, and a state with **two** fields (`message` and `course`) — the key idea being that "multiple state" just means more keys on the *same* schema, not multiple separate objects.

In [ ]:
from typing import Dict, TypedDict 
from langgraph.graph import StateGraph 

`TypedDict` gives the state a typed shape (for editor/type-checker hints — LangGraph doesn't enforce it at runtime). `StateGraph` is the builder you use to assemble nodes and edges before compiling.

In [ ]:
class AgentState(TypedDict): #our state schema
    """A TypedDict for the state of the agent."""
    message: str
    course: str

def greeting_node(state: AgentState) -> AgentState: 
    """ Simple node that add greenings a greeting message to the state"""
    state['message']= "Hey" + state["message"]+ ",you're doing an amzing job learning " + state["course"]
    return state

**Multiple state fields, one schema.** `AgentState` carries two fields, `message` and `course` — both live on the *same* `TypedDict` and both get passed into every node as part of one `state` dict. A node still takes a single `state` argument no matter how many fields the schema has; add a field by adding a key to the `TypedDict`, not by adding a second function parameter.

`greeting_node` mutates `state["message"]` in place and returns the whole dict. For bigger graphs it's usually cleaner to return only the keys you changed, e.g. `return {"message": new_message}` — LangGraph merges that into the full state for you.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("greeter", greeting_node)

graph.set_entry_point("greeter")
graph.set_finish_point("greeter")

app = graph.compile()

**Building the graph:**
- `StateGraph(AgentState)` — create a builder bound to that state schema.
- `add_node(name, fn)` — register a node under a string name.
- `set_entry_point(...)` / `set_finish_point(...)` — mark where execution starts and ends (with one node, it's both).
- `compile()` — turn the builder into a runnable `app` (this is what actually validates the graph and produces something you can `.invoke()`).

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

`app.get_graph().draw_mermaid_png()` renders the compiled graph as a diagram — handy for sanity-checking node wiring before you run it, especially once graphs grow beyond one node.

In [ ]:
result = app.invoke({'message': "Bob", 'course': "Langgraph"})

**Running the graph.** `app.invoke(input_dict)` takes *one* dict with all the initial state values — every field your schema needs (`message` and `course` here) goes into that same dict, not into separate arguments. The optional second positional argument to `invoke` is `config` (thread id, callbacks, etc.), not more state — a common trap when you're first wiring up multiple fields.

In [ ]:
result["message"]